[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/59_diff_attention_solution.ipynb)

# 🔴 Solution: Differential Attention

Reference solution for Differential Attention, which cancels noise by subtracting a scaled second attention map from the first.

Split Q, K each into two halves, compute two softmax attention maps, then return their weighted difference applied to V:

$$\text{DiffAttn}(Q, K, V) = (\text{softmax}(Q_1 K_1^\top / \sqrt{D_h}) - \lambda \cdot \text{softmax}(Q_2 K_2^\top / \sqrt{D_h})) V$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def diff_attention(Q, K, V, lambda_val):
    B, S, D2 = Q.shape
    D_h = D2 // 2
    Q1, Q2 = Q[..., :D_h], Q[..., D_h:]
    K1, K2 = K[..., :D_h], K[..., D_h:]
    scale = D_h ** -0.5
    A1 = torch.softmax(Q1 @ K1.transpose(-2, -1) * scale, dim=-1)
    A2 = torch.softmax(Q2 @ K2.transpose(-2, -1) * scale, dim=-1)
    return (A1 - lambda_val * A2) @ V

In [ ]:
torch.manual_seed(0)
B, S, D2, D_v = 2, 4, 8, 6
Q = torch.randn(B, S, D2)
K = torch.randn(B, S, D2)
V = torch.randn(B, S, D_v)

# lambda=0 should match standard attention on Q1, K1
D_h = D2 // 2
scale = D_h ** -0.5
standard = torch.softmax(Q[..., :D_h] @ K[..., :D_h].transpose(-2, -1) * scale, dim=-1) @ V
diff_zero = diff_attention(Q, K, V, lambda_val=0.0)
print("lambda=0 matches standard attention:", torch.allclose(diff_zero, standard, atol=1e-6))

# lambda=1 with identical halves gives zero output
Q_same = torch.cat([Q[..., :D_h], Q[..., :D_h]], dim=-1)
K_same = torch.cat([K[..., :D_h], K[..., :D_h]], dim=-1)
diff_one = diff_attention(Q_same, K_same, V, lambda_val=1.0)
print("lambda=1 with identical halves gives zero:", torch.allclose(diff_one, torch.zeros_like(diff_one), atol=1e-6))
print("Output shape:", diff_zero.shape)  # (2, 4, 6)

In [ ]:
from torch_judge import check
check("diff_attention")